# Non-Family Credit and Household Enterprise Outcomes — Notebook *Journal of Economic Structures* (MODE FULL)

Data: **IFLS4 (2007) → IFLS5 (2014)**, rumah tangga dengan usaha non-pertanian di 2007.
Treatment: memperoleh pinjaman non-keluarga dalam 12 bulan terakhir (2014). Outcome 2014: **usaha tutup**, laba (IHS), tenaga kerja.

| Bagian | Isi | Tabel naskah |
|---|---|---|
| P | Persiapan data dari file .dta IFLS (otomatis) | – |
| A | Monte Carlo 18 skenario × 500 replikasi | Tabel 2, Lampiran A1, Gambar A1–A4 |
| B | DML-AIPW, heterogenitas, robustness, margin & sensitivitas satuan, penargetan | Tabel 1, 3, 4, 5 |
| D | Uji plasebo 2000–2007 (tanpa & dengan bobot) | Tabel 6 |
| E | Spesifikasi *history-augmented* | Sect. 6.5 |
| C | Ekspor tabel + `angka_untuk_naskah.md` | – |

**Cara menjalankan (batas Kaggle 12 jam):**
1. Settings → **Internet On**, Accelerator **None** (CPU). Tambahkan dataset IFLS (Private) **dan** dataset berisi `umkm_ifls3_placebo.csv` (dari `results.zip` notebook builder IFLS3) lewat *Add Input*.
2. **Sesi 1:** `SHARD = (0, 2)`, lalu **Save Version → Save & Run All (Commit)**.
3. **Sesi 2:** `SHARD = (1, 2)`, commit lagi (bisa paralel dengan sesi 1 di notebook salinan).
4. **Sesi gabungan:** tambahkan *Output* kedua sesi sebagai Input, set `RUN_SIMULATION = False`, commit. Ringkasan Monte Carlo dan semua tabel akan digabung otomatis.

Bagian B (empiris, ±15–30 menit) berjalan di setiap sesi; hasil terakhir yang dipakai.

**Replication settings.** Main empirical specification: `MAIN_SCHEME = "random"`. To re-run the Monte Carlo study set `RUN_SIMULATION = True` and run `SHARD = (0, 2)` and `SHARD = (1, 2)` in two sessions; then run once more with `RUN_SIMULATION = False` and both outputs as input (the notebook merges every `sim_raw_FULL_shard*.csv` it finds).


In [ ]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_v] = "1"   # cegah oversubscription saat paralel
import sys, time, json, math, warnings, itertools, glob, re
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

MODE            = "FULL"      # "FULL" untuk naskah; "FAST" hanya untuk uji kode
RUN_PREP        = True        # bangun umkm_ifls_v2.csv dari .dta IFLS
RUN_SIMULATION  = False     # MC FULL sudah selesai (results 74) -> cukup tambahkan output-nya sebagai Input
RUN_EMPIRICAL   = True
SHARD           = (0, 1)      # sesi 1: (0,2) | sesi 2: (1,2) | satu sesi: (0,1)

IFLS_ROOT  = "/kaggle/input"  # dataset IFLS (Private)
DATA_PATH  = None             # None = pakai hasil Bagian P; atau path umkm_ifls_v2.csv yang sudah ada

N_REPS   = {"FAST": 8, "FULL": 500}[MODE]
N_FOLDS  = 5
N_JOBS   = -1
SEED     = 2026
MAIN_SCHEME = "random"   # spesifikasi utama: fold acak + set C + SE klaster (desain dgn coverage terbaik di Monte Carlo)
PS_CLIP  = 0.01
RUN_CF_ECONML = True

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./output")
for sub in ["sim", "emp", "fig", "tab", "draft"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
DEMO, TAG = False, ""
print("MODE:", MODE, "| N_REPS:", N_REPS, "| SHARD:", SHARD, "| OUT:", OUT)

In [ ]:
# Paket opsional (Kaggle biasanya sudah punya lightgbm; econml perlu internet ON)
try:
    import lightgbm as lgb; HAS_LGB = True
except Exception:
    HAS_LGB = False
HAS_ECONML = False
if RUN_CF_ECONML:
    try:
        import econml; HAS_ECONML = True
    except Exception:
        try:
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "20", "econml"], check=True, timeout=600)
            import econml; HAS_ECONML = True
        except Exception as e:
            print("econml tidak tersedia -> Causal Forest dilewati:", str(e)[:80])
import matplotlib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy import stats
from scipy.optimize import brentq
from scipy.spatial.distance import cdist
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
print("lightgbm:", HAS_LGB, "| econml:", HAS_ECONML)

## 1. Fungsi inti: learner, skema cross-fitting, DML-AIPW, DR-learner, BLP
- **DML-AIPW** (Chernozhukov et al., 2018) dengan skor ortogonal; SE iid dan **SE klaster** (kab/kota).
- **Cross-fitting:** `random` (KFold), `group` (seluruh kab/kota dalam satu fold), `spatial` (blok spasial hasil k-means atas centroid kab/kota, lalu blok dibagi ke fold).
- **Heterogenitas:** DR-learner (Kennedy, 2023) atas pseudo-outcome AIPW, uji **BLP** dan **GATES** (Chernozhukov, Demirer, Duflo & Fernández-Val).

In [ ]:
def make_clf(kind="gbm", fast=False):
    it = 100 if fast else 300
    if kind == "rf":
        return RandomForestClassifier(n_estimators=200 if not fast else 100, min_samples_leaf=10, n_jobs=1, random_state=0)
    if HAS_LGB:
        return lgb.LGBMClassifier(n_estimators=it, learning_rate=0.1 if fast else 0.05, num_leaves=15,
                                  min_child_samples=20, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, n_jobs=1)
    return HistGradientBoostingClassifier(max_iter=it, learning_rate=0.1 if fast else 0.05,
                                          max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0,
                                          early_stopping=False)

def make_reg(kind="gbm", fast=False):
    it = 100 if fast else 300
    if kind == "rf":
        return RandomForestRegressor(n_estimators=200 if not fast else 100, min_samples_leaf=10, n_jobs=1, random_state=0)
    if HAS_LGB:
        return lgb.LGBMRegressor(n_estimators=it, learning_rate=0.1 if fast else 0.05, num_leaves=15,
                                 min_child_samples=20, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, n_jobs=1)
    return HistGradientBoostingRegressor(max_iter=it, learning_rate=0.1 if fast else 0.05,
                                         max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0,
                                         early_stopping=False)

def make_folds(scheme, groups, region_xy, k=5, seed=0, blocks_per_fold=2):
    # groups: label wilayah per observasi; region_xy: DataFrame index=label, kolom [x,y]
    n = len(groups); rng = np.random.default_rng(seed)
    if scheme == "random":
        return list(KFold(k, shuffle=True, random_state=seed).split(np.arange(n)))
    ug = np.asarray(region_xy.index)
    if scheme == "group":
        perm = rng.permutation(len(ug)); fold_of = dict(zip(ug[perm], np.arange(len(ug)) % k))
    elif scheme == "spatial":
        nb = min(len(ug), k * blocks_per_fold)
        lab = KMeans(nb, n_init=5, random_state=seed).fit_predict(region_xy[["x", "y"]].values)
        bperm = rng.permutation(nb); fold_of_block = {b: i % k for i, b in enumerate(bperm)}
        fold_of = {g: fold_of_block[l] for g, l in zip(ug, lab)}
    else:
        raise ValueError(scheme)
    f = np.array([fold_of[g] for g in groups])
    return [(np.where(f != j)[0], np.where(f == j)[0]) for j in range(k) if (f == j).any()]

def cluster_se(psi_centered, groups, w=None):
    # SE rata-rata (berbobot) dari skor terpusat, klaster pada groups
    w = np.ones_like(psi_centered) if w is None else w
    s = pd.Series(w * psi_centered).groupby(np.asarray(groups)).sum().values
    G = len(s)
    return math.sqrt(G / (G - 1) * np.sum(s ** 2)) / np.sum(w)

def _fit_pred(model, Xtr, ytr, Xte, binary):
    if binary:
        if ytr.min() == ytr.max():
            return np.full(len(Xte), float(ytr.mean()))
        return model.fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    return model.fit(Xtr, ytr).predict(Xte)

def dml_aipw(Y, D, X, folds, binary_y, groups, w=None, clip=PS_CLIP, kind="gbm", fast=False):
    n = len(Y); e = np.zeros(n); m1 = np.zeros(n); m0 = np.zeros(n)
    for tr, te in folds:
        e[te] = _fit_pred(make_clf(kind, fast), X[tr], D[tr], X[te], True)
        for d, arr in ((1, m1), (0, m0)):
            sel = tr[D[tr] == d]
            mdl = make_clf(kind, fast) if binary_y else make_reg(kind, fast)
            arr[te] = _fit_pred(mdl, X[sel], Y[sel], X[te], binary_y)
    extreme = float(np.mean((e < clip) | (e > 1 - clip)))
    ec = np.clip(e, clip, 1 - clip)
    psi = m1 - m0 + D * (Y - m1) / ec - (1 - D) * (Y - m0) / (1 - ec)
    w_ = np.ones(n) if w is None else w
    ate = np.sum(w_ * psi) / np.sum(w_)
    se_iid = math.sqrt(np.sum(w_ ** 2 * (psi - ate) ** 2)) / np.sum(w_)
    se_cl = cluster_se(psi - ate, groups, w_)
    # ATT (AIPW)
    p = np.sum(w_ * D) / np.sum(w_)
    num = D * (Y - m0) - (1 - D) * ec / (1 - ec) * (Y - m0)
    att = np.sum(w_ * num) / np.sum(w_ * D)
    att_se = cluster_se((num - D * att) / p, groups, w_)
    return dict(ate=ate, se_iid=se_iid, se_cl=se_cl, att=att, att_se=att_se,
                extreme=extreme, psi=psi, e=ec, m1=m1, m0=m0)

def dr_learner(psi, X, folds, kind="gbm", fast=False):
    tau = np.zeros(len(psi))
    lo, hi = np.quantile(psi, [0.005, 0.995]); y = np.clip(psi, lo, hi)  # winsor ringan utk stabilitas
    for tr, te in folds:
        tau[te] = make_reg(kind, fast).fit(X[tr], y[tr]).predict(X[te])
    return tau

def ols_cluster(y, Xmat, groups, w=None):
    # WLS + SE klaster (CR1). Xmat sudah memuat konstanta.
    w = np.ones(len(y)) if w is None else w
    sw = np.sqrt(w); Xw = Xmat * sw[:, None]; yw = y * sw
    XtX_inv = np.linalg.pinv(Xw.T @ Xw)
    b = XtX_inv @ Xw.T @ yw
    u = yw - Xw @ b
    meat = np.zeros((Xmat.shape[1],) * 2)
    for g in np.unique(groups):
        idx = groups == g; s = Xw[idx].T @ u[idx]; meat += np.outer(s, s)
    n, k = Xmat.shape; G = len(np.unique(groups))
    V = XtX_inv @ meat @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    return b, np.sqrt(np.clip(np.diag(V), 0, None)), G

def blp_test(psi, tau_hat, groups, w=None):
    # psi_i = a + b*(tau_hat_i - mean) ; b>0 & signifikan => heterogenitas terdeteksi
    t = tau_hat - np.average(tau_hat, weights=w)
    Xm = np.column_stack([np.ones_like(t), t])
    b, se, G = ols_cluster(psi, Xm, np.asarray(groups), w)
    pval = 2 * (1 - stats.t.cdf(abs(b[1] / se[1]), df=G - 1)) if se[1] > 0 else np.nan
    return dict(ate_blp=b[0], beta_het=b[1], se_het=se[1], p_het=pval)

def gates(psi, tau_hat, groups, q=5, w=None):
    cut = pd.qcut(tau_hat, q, labels=False, duplicates="drop")
    rows = []
    for g in np.unique(cut):
        i = cut == g; ww = None if w is None else w[i]
        m = np.average(psi[i], weights=ww)
        rows.append(dict(group=int(g) + 1, gate=m, se=cluster_se(psi[i] - m, np.asarray(groups)[i], ww), n=int(i.sum())))
    return pd.DataFrame(rows)
print("Fungsi inti siap.")

# BAGIAN P — Persiapan data IFLS 2007 → 2014
Membangun `umkm_ifls_v2.csv`. Centroid kab/kota (kode BPS, 514 kab/kota) sudah tertanam di sel ini, jadi tidak perlu unduhan.
Perbaikan yang sudah termasuk: omzet & izin dibuang (hampir kosong/konstan), kode missing pekerja 9999 dibersihkan + winsor p99,5, pendidikan kepala RT dikonversi ke tahun sekolah (cek dengan codebook `bk_ar1`).

In [ ]:
if RUN_PREP:
    ROOT = IFLS_ROOT
    def find(folder_hint, fname):
        hits = [f for f in glob.glob(f"{ROOT}/**/{fname}.dta", recursive=True) if folder_hint in f]
        if not hits:
            raise FileNotFoundError(f"{fname}.dta di folder '{folder_hint}' tidak ditemukan")
        return sorted(hits, key=len)[0]
    def rd(folder_hint, fname, cols=None):
        df = pd.read_stata(find(folder_hint, fname), convert_categoricals=False)
        df.columns = [c.lower() for c in df.columns]
        if cols:
            df = df[[c for c in cols if c in df.columns]]
        return df
    def sid(s): return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    def num(s): return pd.to_numeric(s, errors="coerce")
    ihs = np.arcsinh

    # 1. SAMPEL 2007: RT yang punya usaha non-pertanian
    # ---------------------------------------------------------------------
    nt1_07 = rd("hh07_b2", "b2_nt1", ["hhid07", "nt01"])
    nt1_07["hhid07"] = sid(nt1_07.hhid07)
    base = nt1_07.loc[num(nt1_07.nt01) == 1, ["hhid07"]].drop_duplicates()
    print("RT dengan usaha non-pertanian 2007:", len(base))

    # karakteristik usaha 2007 (agregat per RT, usaha yang masih beroperasi)
    nt2_07 = rd("hh07_b2", "b2_nt2")
    nt2_07["hhid07"] = sid(nt2_07.hhid07)
    op07 = nt2_07[num(nt2_07.get("nt18")).fillna(1) == 1].copy()
    for c in ["nt07", "nt09", "nt22", "nt23", "nt10a", "nt10b", "nt10c1", "nt10c4", "nt10h", "nt15myr", "nt05b", "nt05d"]:
        op07[c] = num(op07[c]) if c in op07 else np.nan
    op07["aset"] = op07[["nt10a", "nt10b", "nt10c1", "nt10c4", "nt10h"]].sum(axis=1, min_count=1)
    op07["pekerja"] = op07[["nt22", "nt23"]].sum(axis=1, min_count=1)
    op07["umur"] = 2007 - op07["nt15myr"].where(op07["nt15myr"].between(1900, 2007))
    g07 = op07.groupby("hhid07").agg(
        omzet07=("nt07", "sum"), laba07=("nt09", "sum"), pekerja07=("pekerja", "sum"),
        aset07=("aset", "sum"), umur_usaha07=("umur", "max"), n_usaha07=("nt07", "size"),
        usaha_di_rumah07=("nt05b", lambda s: float((s == 1).any())),
        izin07=("nt05d", lambda s: float(s.notna().any() and (s.fillna(0) > 0).any())),
    ).reset_index()
    base = base.merge(g07, on="hhid07", how="left")

    # pinjaman 2007 (kredit sebelumnya) — kovariat pra-treatment
    bh07 = rd("hh07_b2", "b2_bh", ["hhid07", "bh07", "bh10"])
    bh07["hhid07"] = sid(bh07.hhid07)
    bh07["pinjam07"] = (num(bh07.bh07) == 1).astype(float)
    bh07["ihs_pinjam07"] = ihs(num(bh07.bh10).fillna(0) * bh07["pinjam07"])
    base = base.merge(bh07[["hhid07", "pinjam07", "ihs_pinjam07"]], on="hhid07", how="left")

    # pengeluaran per kapita 2007
    pce = rd("pce", "pce07nom")
    pce["hhid07"] = sid(pce.hhid07)
    pce_col = next((c for c in pce.columns if re.fullmatch(r"(pce|pcexp|pce07|hhexp_pc|pcexpend)", c)), None)
    if pce_col is None:   # fallback: total pengeluaran / hhsize
        tot = next((c for c in pce.columns if re.fullmatch(r"(hhexp|totexp|hhexp07|xtotal|texp)", c)), None)
        print("Kolom PCE tidak ditemukan otomatis; kandidat:", [c for c in pce.columns if "exp" in c or "pce" in c][:20])
        pce["pce07"] = num(pce[tot]) / num(pce["hhsize"]) if tot else np.nan
    else:
        pce["pce07"] = num(pce[pce_col])
    pce["hhsize07"] = num(pce.get("hhsize"))
    base = base.merge(pce[["hhid07", "pce07", "hhsize07"]], on="hhid07", how="left")

    # kepala RT 2007 (Book K, roster)
    try:
        ar = rd("hh07_bk", "bk_ar1", ["hhid07", "ar02b", "ar07", "ar09", "ar16"])
        ar["hhid07"] = sid(ar.hhid07)
        head = ar[num(ar.ar02b) == 1].drop_duplicates("hhid07")
        head = head.assign(kepala_perempuan=(num(head.ar07) == 3).astype(float),   # IFLS: 1=laki,3=perempuan
                           usia_kepala=num(head.ar09).where(num(head.ar09).between(10, 100)),
                           pendidikan_kepala=num(head.ar16).where(num(head.ar16) < 90))
        base = base.merge(head[["hhid07", "kepala_perempuan", "usia_kepala", "pendidikan_kepala"]], on="hhid07", how="left")
    except Exception as e:
        print("Roster kepala RT dilewati:", e)

    # lokasi & desa 2007
    sc07 = rd("hh07_bk", "bk_sc", ["hhid07", "sc05"])
    sc07["hhid07"] = sid(sc07.hhid07)
    sc07["perkotaan07"] = (num(sc07.sc05) == 1).astype(float)
    base = base.merge(sc07[["hhid07", "perkotaan07"]].drop_duplicates("hhid07"), on="hhid07", how="left")

    # ---------------------------------------------------------------------
    # 2. LACAK KE 2014 (htrack IFLS5)
    # ---------------------------------------------------------------------
    ht = rd("hh14_trk", "htrack", ["hhid07", "hhid14", "commid07", "result14", "hwt14la", "hwt14l_",
                                    "sc01_14_14", "sc02_14_14", "mover14", "splitoff14"])
    ht["hhid07"] = sid(ht.hhid07); ht["hhid14"] = sid(ht.hhid14)
    ht = ht[(ht.hhid14 != "") & (ht.hhid14 != "nan")]
    # satu RT 2014 per RT 2007: utamakan RT asal (bukan split-off)
    if "splitoff14" in ht:
        ht["_ord"] = num(ht.splitoff14).fillna(0)
    else:
        ht["_ord"] = 0
    ht = ht.sort_values(["hhid07", "_ord", "hhid14"]).drop_duplicates("hhid07")
    base = base.merge(ht, on="hhid07", how="inner")
    print("Terlacak di 2014:", len(base))

    # desa 2007: jumlah lembaga kredit di desa (gtype dengan g3a==1) & jarak terdekat
    try:
        g = rd("cf07_bk1", "bk1_g", ["commid07", "gtype", "g3a", "g3b"])
        g["commid07"] = sid(g.commid07)
        agg = g.assign(ada=(num(g.g3a) == 1).astype(float), km=num(g.g3b).where(num(g.g3b) < 900)).groupby("commid07").agg(
            n_lembaga_kredit_desa07=("ada", "sum"), jarak_lembaga_min07=("km", "min")).reset_index()
        base["commid07"] = sid(base.commid07)
        base = base.merge(agg, on="commid07", how="left")
        base["jarak_lembaga_min07"] = base["jarak_lembaga_min07"].fillna(0)  # ada di desa -> 0 km
        print("Catatan: cek kode g3a di kuesioner CF07 (asumsi 1 = ada di desa).")
    except Exception as e:
        print("Data desa dilewati:", e)

    # ---------------------------------------------------------------------
    # 3. TREATMENT 2014: memperoleh pinjaman non-keluarga 12 bln terakhir
    # ---------------------------------------------------------------------
    bh14 = rd("hh14_b2", "b2_bh", ["hhid14", "bh02", "bh07", "bh10"])
    bh14["hhid14"] = sid(bh14.hhid14)
    bh14["kredit"] = (num(bh14.bh07) == 1).astype(int)            # gagal/tdk mencoba = 0
    bh14["mencoba_pinjam"] = (num(bh14.bh02) == 1).astype(int)
    bh14["ihs_jumlah_pinjam14"] = ihs(num(bh14.bh10).fillna(0) * bh14.kredit)
    base = base.merge(bh14[["hhid14", "kredit", "mencoba_pinjam", "ihs_jumlah_pinjam14"]], on="hhid14", how="inner")

    # ---------------------------------------------------------------------
    # 4. OUTCOME 2014
    # ---------------------------------------------------------------------
    nt1_14 = rd("hh14_b2", "b2_nt1", ["hhid14", "nt01"]); nt1_14["hhid14"] = sid(nt1_14.hhid14)
    nt2_14 = rd("hh14_b2", "b2_nt2"); nt2_14["hhid14"] = sid(nt2_14.hhid14)
    for c in ["nt07", "nt09", "nt22", "nt23", "nt18"]:
        nt2_14[c] = num(nt2_14[c]) if c in nt2_14 else np.nan
    op14 = nt2_14[nt2_14.nt18.fillna(1) == 1].copy()                 # masih berproduksi
    op14["pekerja"] = op14[["nt22", "nt23"]].sum(axis=1, min_count=1)
    g14 = op14.groupby("hhid14").agg(omzet14=("nt07", "sum"), laba14=("nt09", "sum"),
                                     pekerja14=("pekerja", "sum"), n_usaha14=("nt07", "size")).reset_index()
    base = base.merge(nt1_14, on="hhid14", how="left").merge(g14, on="hhid14", how="left")
    base["usaha_tutup"] = ((num(base.nt01) != 1) | base.n_usaha14.isna()).astype(int)
    for c in ["omzet14", "laba14", "pekerja14"]:
        base[c] = base[c].where(base.usaha_tutup == 0, 0)            # usaha tutup -> 0
    base["ihs_omzet"] = ihs(base.omzet14)
    base["ihs_laba"] = ihs(base.laba14)
    base["tenaga_kerja"] = base.pekerja14


    # 5. WILAYAH + centroid kab/kota (tertanam; sumber: data BPS via ArrayAccess, MIT)
    CENT_STR = "1101,2.5383,96.0382;1102,2.3069,97.6371;1103,3.0700,97.4242;1104,3.3380,97.6175;1105,4.7028,97.6392;1106,4.5625,96.8149;1107,4.4521,96.1857;1108,5.4087,95.4260;1109,5.1132,96.0868;1110,5.0836,96.6244;1111,4.9936,97.1507;1112,3.8357,96.8673;1113,3.9810,97.3235;1114,4.2038,98.0010;1115,4.1759,96.5056;1116,4.8108,95.6665;1117,4.7748,96.9924;1118,5.1048,96.1952;1171,5.5618,95.3287;1172,5.9243,95.2456;1173,4.4848,97.9816;1174,5.1620,97.1027;1175,2.7561,97.9417;1201,1.0843,97.7301;1202,0.7796,99.3906;1203,1.5511,99.1780;1204,1.8678,98.5962;1205,2.0112,99.0688;1206,2.3856,99.2467;1207,2.2702,100.0132;1208,2.8469,99.5330;1209,2.9506,99.0493;1210,2.8354,98.2544;1211,3.0996,98.2497;1212,3.4799,98.7071;1213,3.7602,98.2326;1214,0.1980,98.2336;1215,2.2383,98.6071;1216,2.5303,98.2685;1217,2.5551,98.6968;1218,3.3492,99.0314;1219,3.2297,99.4800;1220,1.6364,99.8455;1221,1.1565,99.8015;1222,1.8338,100.0605;1223,2.4256,99.7342;1224,1.3102,97.2902;1225,0.9823,97.4580;1271,1.7308,98.7841;1272,2.9728,99.7954;1273,2.9546,99.0635;1274,3.3281,99.1541;1275,3.6424,98.6695;1276,3.6002,98.4887;1277,1.3869,99.2858;1278,1.2520,97.5805;1301,-2.4611,99.8254;1302,-1.7178,100.7905;1303,-0.9275,100.8379;1304,-0.6487,101.1429;1305,-0.4698,100.5828;1306,-0.5667,100.2098;1307,-0.2532,100.1538;1308,0.0291,100.5559;1309,0.3999,100.0475;1310,-1.3645,101.2736;1311,-1.2506,101.5193;1312,0.1880,99.6077;1371,-0.9364,100.3315;1372,-0.7789,100.6151;1373,-0.6615,100.7566;1374,-0.4695,100.4013;1375,-0.2992,100.3691;1376,-0.2295,100.6328;1377,-0.6120,100.1284;1401,-0.4938,101.4766;1402,-0.4926,102.2485;1403,-0.2924,103.1848;1404,0.1625,102.4358;1405,0.8280,101.9038;1406,0.3203,101.0614;1407,0.8933,100.4905;1408,1.5252,101.7206;1409,2.0709,100.8054;1410,1.0520,102.7216;1471,0.5547,101.4595;1473,1.8532,101.3687;1501,-2.0612,101.4830;1502,-2.2071,102.0862;1503,-2.3323,102.6426;1504,-1.8531,102.9884;1505,-1.7049,103.7826;1506,-1.2662,103.9221;1507,-1.0927,103.1098;1508,-1.3952,102.3232;1509,-1.5158,101.9783;1571,-1.6204,103.6031;1572,-2.1383,101.3489;1601,-4.1105,104.0873;1602,-3.3266,105.4149;1603,-3.6949,103.9978;1604,-3.8770,103.3436;1605,-3.1752,103.0679;1606,-2.4958,103.8304;1607,-2.3987,104.8173;1608,-4.5677,103.8846;1609,-4.1063,104.5414;1610,-3.4197,104.5600;1611,-3.7170,102.9066;1612,-3.2084,103.9321;1613,-2.7062,102.7170;1671,-2.9794,104.7442;1672,-3.4547,104.2383;1673,-4.1214,103.2684;1674,-3.2538,102.8621;1701,-4.3589,103.0442;1702,-3.4382,102.6831;1703,-4.1158,101.9987;1704,-4.5906,103.4168;1705,-4.0804,102.6345;1706,-2.7306,101.4442;1707,-3.0526,102.2138;1708,-3.6475,102.6278;1709,-3.6778,102.4099;1771,-3.8360,102.3265;1801,-5.0770,104.0758;1802,-5.5217,104.7609;1803,-5.6699,105.4688;1804,-5.1205,105.7437;1805,-4.8741,105.1760;1806,-4.8084,104.8118;1807,-4.5733,104.6774;1808,-4.3987,105.5376;1809,-5.4821,105.1385;1810,-5.3691,104.9174;1811,-3.9739,105.4038;1812,-4.4329,105.0978;1813,-5.4092,104.1510;1871,-5.4287,105.2675;1872,-5.1184,105.3106;1901,-1.9359,105.9418;1902,-2.9396,107.5498;1903,-1.8402,105.4470;1904,-2.4328,106.2993;1905,-2.7861,106.4907;1906,-2.9549,108.2993;1971,-2.1093,106.1111;2101,0.8357,103.6330;2102,0.9133,105.9079;2103,3.6266,108.4435;2104,-0.3940,104.9327;2105,2.9036,106.0411;2171,0.8344,104.0132;2172,0.9150,104.4447;3101,-5.6122,106.6155;3171,-6.2841,106.8021;3172,-6.2611,106.9051;3173,-6.1823,106.8366;3174,-6.1602,106.7573;3175,-6.1361,106.8419;3201,-6.5454,106.8141;3202,-7.0765,106.7177;3203,-7.0538,107.1304;3204,-7.0635,107.5912;3205,-7.3429,107.7782;3206,-7.4279,108.1735;3207,-7.3113,108.4466;3208,-6.9876,108.5869;3209,-6.7614,108.5787;3210,-6.8077,108.2277;3211,-6.8102,107.9804;3212,-6.4536,108.1955;3213,-6.5007,107.7283;3214,-6.5924,107.4113;3215,-6.2635,107.3640;3216,-6.1979,107.1326;3217,-6.9053,107.4542;3218,-7.6405,108.5567;3271,-6.5952,106.7919;3272,-6.9357,106.9158;3273,-6.9033,107.6426;3274,-6.7437,108.5571;3275,-6.2850,106.9701;3276,-6.3878,106.8178;3277,-6.8823,107.5433;3278,-7.3600,108.2313;3279,-7.3805,108.5661;3301,-7.4618,108.9757;3302,-7.4550,109.1686;3303,-7.3273,109.4041;3304,-7.3560,109.6396;3305,-7.6439,109.6093;3306,-7.7080,109.9700;3307,-7.3998,109.9057;3308,-7.5136,110.2448;3309,-7.3871,110.6194;3310,-7.6748,110.6229;3311,-7.6828,110.8314;3312,-7.9621,111.0367;3313,-7.6196,110.9514;3314,-7.3923,110.9621;3315,-7.0979,110.8942;3316,-7.1106,111.3682;3317,-6.7699,111.4624;3318,-6.7049,111.0371;3319,-6.7976,110.8680;3320,-6.2610,110.5697;3321,-6.9229,110.6511;3322,-7.2878,110.4566;3323,-7.2399,110.1315;3324,-7.0235,110.1394;3325,-7.0317,109.8634;3326,-7.0424,109.6411;3327,-7.0119,109.3924;3328,-7.0554,109.1559;3329,-7.0548,108.9342;3371,-7.4726,110.2198;3372,-7.5592,110.8190;3373,-7.3374,110.5016;3374,-7.0231,110.3872;3375,-6.8937,109.6783;3376,-6.8698,109.1183;3401,-7.8126,110.1387;3402,-7.8980,110.3626;3403,-7.9931,110.5819;3404,-7.6897,110.3813;3471,-7.8033,110.3748;3501,-8.1019,111.1627;3502,-7.9712,111.5294;3503,-8.1384,111.6189;3504,-8.0730,111.9167;3505,-8.1317,112.2173;3506,-7.8016,112.1096;3507,-8.1126,112.6235;3508,-8.1246,113.1395;3509,-8.2656,113.6529;3510,-8.3325,114.2210;3511,-7.9453,113.9360;3512,-7.7987,114.0203;3513,-7.8520,113.2857;3514,-7.7506,112.8352;3515,-7.4534,112.6646;3516,-7.5411,112.5017;3517,-7.5615,112.2592;3518,-7.6151,111.9473;3519,-7.6118,111.6487;3520,-7.6534,111.3474;3521,-7.4325,111.3948;3522,-7.2278,111.7947;3523,-6.9594,111.8960;3524,-7.1236,112.3125;3525,-6.5616,112.5716;3526,-7.0462,112.9106;3527,-7.1013,113.2619;3528,-7.0708,113.5035;3529,-6.1507,114.9238;3571,-7.8230,112.0179;3572,-8.0962,112.1665;3573,-7.9813,112.6319;3574,-7.7712,113.2026;3575,-7.6535,112.9119;3576,-7.4707,112.4373;3577,-7.6378,111.5275;3578,-7.2746,112.7192;3579,-7.8337,112.5342;3601,-6.6192,105.6378;3602,-6.6433,106.1946;3603,-6.1832,106.5319;3604,-6.0721,106.1186;3671,-6.1767,106.6500;3672,-5.9784,106.0048;3673,-6.1003,106.1606;3674,-6.2956,106.7083;5101,-8.3137,114.6829;5102,-8.4389,115.0663;5103,-8.5456,115.1676;5104,-8.4819,115.2983;5105,-8.6374,115.4920;5106,-8.3340,115.3450;5107,-8.3586,115.5510;5108,-8.2225,114.9450;5171,-8.6721,115.2243;5201,-8.6664,116.0800;5202,-8.6824,116.2691;5203,-8.5916,116.5602;5204,-8.6167,117.5864;5205,-8.4938,118.1167;5206,-8.4833,118.5506;5207,-8.8003,116.9011;5208,-8.3454,116.2611;5271,-8.5887,116.1201;5272,-8.4408,118.7970;5301,-9.5853,119.3433;5302,-9.8053,120.2601;5303,-9.8124,123.7487;5304,-9.8225,124.4374;5305,-9.3406,124.4737;5306,-9.1758,124.9929;5307,-8.3335,124.4754;5308,-8.3754,123.5660;5309,-8.3502,122.9940;5310,-8.4570,122.1818;5311,-8.6736,121.7103;5312,-8.6499,120.9972;5313,-8.5784,120.3873;5314,-10.7192,123.0410;5315,-8.5740,119.8714;5316,-9.5938,119.6710;5317,-9.5440,119.1703;5318,-8.6744,121.3098;5319,-8.5738,120.7091;5320,-10.6241,121.6389;5321,-9.5463,124.8658;5371,-10.2114,123.6054;6101,1.5227,109.3267;6102,1.0331,109.4144;6103,0.5109,109.7047;6104,0.3473,109.1109;6105,0.3507,110.4443;6106,-1.6967,110.5904;6107,0.1938,112.1090;6108,0.8341,112.8766;6109,0.0537,110.9252;6110,-0.7351,111.7872;6111,-1.2477,109.5745;6112,-0.3790,109.5132;6171,-0.0297,109.3281;6172,0.8910,109.0256;6201,-2.5483,111.6777;6202,-2.2299,112.6776;6203,-1.9180,114.2221;6204,-1.9640,115.0345;6205,-0.7397,115.1521;6206,-2.5604,111.0673;6207,-1.8232,111.3394;6208,-2.1205,112.1289;6209,-1.8755,112.9332;6210,-2.5054,113.9549;6211,-0.9869,113.5131;6212,-2.0149,115.1746;6213,-0.0444,114.1713;6271,-2.0112,113.7944;6301,-3.8436,114.9471;6302,-3.5253,116.0356;6303,-3.2794,115.0512;6304,-3.0302,114.6042;6305,-2.8680,115.1551;6306,-2.7181,115.2329;6307,-2.6115,115.5242;6308,-2.4298,115.1315;6309,-1.8370,115.4539;6310,-3.4453,115.6673;6311,-2.2852,115.5865;6371,-3.3247,114.5908;6372,-3.4839,114.8141;6401,-1.6877,116.1072;6402,-0.4712,115.8719;6403,0.1950,116.5450;6404,0.9306,117.4977;6405,1.8151,117.5824;6409,-1.2015,116.6453;6411,0.6657,114.8282;6471,-1.1608,116.8693;6472,-0.5097,117.1755;6474,0.1118,117.4592;6501,2.5823,115.6987;6502,2.8469,116.9876;6503,3.5540,117.2598;6504,3.9055,116.7430;6571,3.3410,117.5919;7101,0.7207,124.0349;7102,1.2416,124.8143;7103,3.9015,125.5268;7104,4.6467,126.8704;7105,1.0711,124.5282;7106,1.5872,124.9597;7107,0.7602,123.4258;7108,2.4404,125.3288;7109,0.9817,124.7504;7110,0.4545,123.9786;7111,0.7113,124.5177;7171,1.5393,124.8074;7172,1.4914,125.1634;7173,1.3293,124.8166;7174,0.7317,124.3132;7201,-1.3992,123.1728;7202,-1.0486,122.6506;7203,-2.8946,122.3102;7204,-1.6737,120.5297;7205,-0.3510,119.8251;7206,0.9812,120.6571;7207,0.9942,121.5189;7208,-0.2392,120.6257;7209,-0.8729,121.6428;7210,-1.4616,120.0028;7211,-1.8596,123.6147;7212,-1.8467,121.4469;7271,-0.7896,119.8993;7301,-6.6272,121.0549;7302,-5.4841,120.2173;7303,-5.4715,119.9777;7304,-5.5448,119.7120;7305,-5.4109,119.4185;7306,-5.3301,119.6979;7307,-5.1973,120.1784;7308,-4.9651,119.7197;7309,-6.1241,118.4245;7310,-4.4338,119.6967;7311,-4.6800,120.0890;7312,-4.3186,119.9012;7313,-3.9682,120.1641;7314,-3.8081,119.9979;7315,-3.6486,119.6077;7316,-3.5321,119.8923;7317,-3.1666,120.1465;7318,-3.0569,119.7071;7322,-2.4063,120.1421;7325,-2.5321,121.1310;7326,-2.8991,119.8803;7371,-5.1196,119.4026;7372,-4.0203,119.6661;7373,-2.9868,120.1406;7401,-5.3582,122.9524;7402,-4.8711,122.6226;7403,-3.5129,122.0422;7404,-4.1022,121.4555;7405,-4.2601,122.4079;7406,-4.9248,121.8078;7407,-5.6862,124.0376;7408,-3.2634,121.1633;7409,-4.7535,123.0208;7410,-3.4195,121.9572;7411,-3.8204,121.5673;7412,-4.1194,123.0973;7413,-4.7708,122.4145;7414,-5.3326,122.2908;7415,-5.8513,122.6450;7471,-4.0002,122.5455;7472,-5.4232,122.6628;7501,0.6816,122.3662;7502,0.7021,122.6627;7503,0.7026,121.6479;7504,0.5562,123.2944;7505,0.8651,122.6284;7571,0.5345,123.0476;7601,-3.2428,118.9365;7602,-3.3026,119.1953;7603,-2.9964,119.3237;7604,-2.4270,118.4332;7605,-1.3421,119.5764;7606,-2.0210,119.5019;8101,-7.5004,131.3803;8102,-5.6551,132.8096;8103,-3.6654,129.0067;8104,-3.3449,126.6908;8105,-6.2126,134.4798;8106,-2.4736,128.2037;8107,-3.8788,130.8795;8108,-7.2893,127.9056;8109,-3.5084,126.6196;8171,-3.6837,128.1649;8172,-5.4490,132.3714;8201,1.3589,127.5930;8202,0.2211,128.7130;8203,-2.1144,125.9074;8204,-0.7457,127.8130;8205,1.5595,127.8349;8206,1.0029,128.2248;8207,2.3040,128.4094;8208,-1.8256,124.8052;8271,0.8929,126.7835;8272,0.3901,127.5968;9101,-3.2030,132.8386;9102,-3.5483,134.0348;9103,-2.6384,134.3866;9104,-2.0870,133.3224;9105,-1.0011,133.8368;9106,-1.6660,132.1058;9107,-1.0553,131.4567;9108,-0.8272,130.3227;9109,-0.8096,132.7273;9110,-1.4242,132.3680;9111,-1.6072,133.9660;9112,-1.2018,133.7208;9171,-0.8661,131.3094;9401,-7.8693,139.3298;9402,-4.1226,138.9776;9403,-3.0362,140.0378;9404,-3.2064,135.4966;9408,-1.7200,136.0321;9409,-1.0129,135.7461;9410,-3.7578,136.5823;9411,-3.4894,137.8449;9412,-4.6299,136.3425;9413,-6.1536,140.3222;9414,-6.2475,139.3166;9415,-5.5826,138.5974;9416,-4.3891,139.5433;9417,-4.4947,140.4785;9418,-3.4888,138.4035;9419,-2.5324,139.0662;9420,-3.3267,140.6530;9426,-2.7293,136.6967;9427,-0.0150,135.0496;9428,-2.4436,137.9762;9429,-4.4362,138.2188;9430,-3.9573,138.2032;9431,-3.6256,139.0325;9432,-3.7851,139.4358;9433,-3.5681,137.4168;9434,-3.9994,135.6373;9435,-3.4086,136.7425;9436,-4.1557,136.4082;9471,-2.6514,140.8070"
    cent = pd.DataFrame([r.split(",") for r in CENT_STR.split(";")], columns=["kab_kode", "lat", "lon"]).astype(float)
    cent["kab_kode"] = cent.kab_kode.astype("int64")
    p = num(base.sc01_14_14); k = num(base.sc02_14_14)
    base["prov_kode"] = p.astype("Int64")
    base["kab_kode"] = np.where(k < 100, p * 100 + k, k).astype("int64")
    base = base.merge(cent, on="kab_kode", how="left")
    base["lat_imputed"] = base.lat.isna().astype(int)
    for c_ in ["lat", "lon"]:
        base[c_] = base[c_].fillna(base.groupby("prov_kode")[c_].transform("mean"))
    print(f"Centroid kab/kota cocok: {1 - base.lat_imputed.mean():.1%} observasi")

    # 6. FINAL + perbaikan v2
    base["weight"] = num(base.hwt14la).fillna(num(base.get("hwt14l_")))
    base["weight"] = base["weight"].fillna(base["weight"].median())
    base["ihs_laba07"] = ihs(base.laba07.fillna(0)); base["ihs_aset07"] = ihs(base.aset07.fillna(0))
    base["log_pce07"] = np.log(base.pce07.where(base.pce07 > 0))
    base["pindah"] = (num(base.mover14).fillna(0) > 0).astype(int)
    base["laba_rp"] = base.laba14                     # laba 2014 (Rp), 0 bila tutup
    base["id"] = base.hhid07
    base.loc[base.pekerja07 >= 999, "pekerja07"] = np.nan
    EDU = {1: 0, 90: 0, 2: 6, 11: 6, 17: 6, 72: 6, 14: 6, 3: 9, 4: 9, 12: 9, 73: 9,
           5: 12, 6: 12, 15: 12, 74: 12, 60: 14, 13: 16, 61: 16, 62: 18, 63: 21}
    if "pendidikan_kepala" in base:
        base["pendidikan_kepala"] = base.pendidikan_kepala.map(EDU)
    keep = ["id", "hhid07", "hhid14", "kab_kode", "prov_kode", "lat", "lon", "lat_imputed", "weight",
            "kredit", "mencoba_pinjam", "ihs_laba", "laba_rp", "tenaga_kerja", "usaha_tutup",
            "ihs_laba07", "pekerja07", "ihs_aset07", "umur_usaha07", "n_usaha07", "usaha_di_rumah07",
            "pinjam07", "ihs_pinjam07", "log_pce07", "hhsize07", "kepala_perempuan", "usia_kepala",
            "pendidikan_kepala", "perkotaan07", "n_lembaga_kredit_desa07", "jarak_lembaga_min07", "pindah"]
    df = base[[c_ for c_ in keep if c_ in base.columns]].copy()
    COVS_BASE = ["ihs_laba07", "pekerja07", "ihs_aset07", "umur_usaha07", "n_usaha07", "usaha_di_rumah07",
                 "pinjam07", "ihs_pinjam07", "log_pce07", "hhsize07", "kepala_perempuan", "usia_kepala",
                 "pendidikan_kepala", "perkotaan07", "n_lembaga_kredit_desa07", "jarak_lembaga_min07"]
    for c_ in [c_ for c_ in COVS_BASE if c_ in df]:
        if df[c_].isna().any():
            df[f"{c_}_miss"] = df[c_].isna().astype(int); df[c_] = df[c_].fillna(df[c_].median())
    for c_ in ["pekerja07", "tenaga_kerja"]:
        df[c_] = df[c_].clip(upper=df[c_].quantile(0.995))
    DATA_PATH = os.path.join(OUT, "umkm_ifls_v2.csv")
    df.to_csv(DATA_PATH, index=False)
    print(f"N={len(df):,} | kab/kota={df.kab_kode.nunique()} | provinsi={df.prov_kode.nunique()} | "
          f"kredit={df.kredit.mean():.1%} | usaha tutup={df.usaha_tutup.mean():.1%} -> {DATA_PATH}")
elif DATA_PATH is None:
    hits = glob.glob(f"{OUT}/umkm_ifls_v2.csv") + glob.glob("/kaggle/input/**/umkm_ifls_v2.csv", recursive=True)
    DATA_PATH = hits[0] if hits else None
    print("DATA_PATH:", DATA_PATH)

# BAGIAN A — Monte Carlo (bukti metodologis)

**DGP.** G wilayah pada bidang [0,1]²; konfounder wilayah tak teramati $U_g$ dari proses Gauss (kovarians eksponensial, *length-scale* 0,3) → ketergantungan spasial. Kekuatan konfounding spasial = `rho`; kekuatan seleksi/overlap = `kappa`.

- Treatment (akses KUR): $\Pr(D=1)=\Lambda(\kappa\,h(X)+0.8\rho U_g+c_D)$, ±40% penerima.
- Outcome langka (usaha tutup, **terdefinisi untuk semua UMKM**): $\Pr(Y=1)=\Lambda(a+f(X)+0.8\rho U_g+D\,\tau(X))$, dengan $\tau(X)=-0.5-0.4X_1$ (KUR menurunkan risiko tutup, lebih besar pada usaha tertentu). $a$ dikalibrasi agar prevalensi = 3%, 5%, atau 10%.
- Estimand: ATE selisih risiko sampel $\frac1n\sum_i[p_{1i}-p_{0i}]$.

**Spesifikasi kovariat:** A = X; B = X + koordinat; C = X + koordinat + rata-rata wilayah (Mundlak, *leave-one-out*).

**Desain (one-factor-at-a-time, disilangkan dengan rho ∈ {0,1,2}):** baseline (n=2000, G=50, prev=5%, kappa=1); variasi prevalensi {3,5,10}%; variasi (n,G) {(1000,35),(2000,50),(5000,100)}; variasi kappa {1, 2.5}. Total 18 skenario. Baseline: 3 skema cross-fitting × 3 spesifikasi; blok lain: 3 skema × spesifikasi C (hemat waktu). Pembanding: naif dan OLS.

**Estimasi waktu FULL (500 rep, 4 CPU Kaggle):** lihat cetakan waktu per skenario. Bila total > 12 jam, pakai `SHARD=(0,2)` lalu `(1,2)` di sesi kedua, kemudian lampirkan `sim_raw_*.csv` sesi pertama sebagai *dataset input* agar digabung otomatis.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

def simulate(n, G, rho, kappa, prev, rng, ls=0.3, p=6):
    xy = rng.uniform(0, 1, (G, 2))
    K = np.exp(-cdist(xy, xy) / ls) + 1e-8 * np.eye(G)
    U = np.linalg.cholesky(K) @ rng.normal(size=G); U = (U - U.mean()) / U.std()
    share = rng.dirichlet(np.full(G, 3.0))
    g = rng.choice(G, size=n, p=share)
    g[:G] = np.arange(G) if n >= G else g[:G]            # tiap wilayah minimal 1 UMKM
    X = rng.normal(size=(n, p)); X[:, 0] += 0.5 * U[g]   # kovariat teramati ikut mencerminkan kondisi wilayah
    loc = xy[g] + rng.normal(0, 0.02, (n, 2))
    h = 0.6 * X[:, 0] - 0.5 * X[:, 1] + 0.4 * X[:, 2] * (X[:, 3] > 0) + 0.3 * np.tanh(X[:, 4])
    lin_d = kappa * h + 0.8 * rho * U[g]
    cD = brentq(lambda c: sigmoid(lin_d + c).mean() - 0.40, -20, 20)
    D = rng.binomial(1, sigmoid(lin_d + cD))
    f = 0.5 * X[:, 0] + 0.4 * np.sin(2 * X[:, 2]) + 0.3 * (X[:, 4] ** 2 - 1) - 0.3 * X[:, 5] + 0.8 * rho * U[g]
    tau_l = -0.5 - 0.4 * X[:, 1]
    a = brentq(lambda a: sigmoid(a + f + D * tau_l).mean() - prev, -30, 10)
    p1, p0 = sigmoid(a + f + tau_l), sigmoid(a + f)
    Y = rng.binomial(1, np.where(D == 1, p1, p0))
    return dict(X=X, loc=loc, g=g, D=D, Y=Y.astype(float), tau=p1 - p0, ate=float(np.mean(p1 - p0)),
                region_xy=pd.DataFrame(xy, columns=["x", "y"], index=np.arange(G)))

def mundlak(X, g):
    df = pd.DataFrame(X); s = df.groupby(g).transform("sum"); c = pd.Series(g).map(pd.Series(g).value_counts()).values
    loo = (s.values - X) / np.maximum(c - 1, 1)[:, None]
    return np.column_stack([loo, np.log(c)])

def features(sim, spec):
    if spec == "A": return sim["X"]
    if spec == "B": return np.column_stack([sim["X"], sim["loc"]])
    return np.column_stack([sim["X"], sim["loc"], mundlak(sim["X"], sim["g"])])

def one_rep(sc, rep):
    ss = np.random.SeedSequence([SEED, sc["sid"], rep]); rng = np.random.default_rng(ss)
    sim = simulate(sc["n"], sc["G"], sc["rho"], sc["kappa"], sc["prev"], rng)
    Y, D, g = sim["Y"], sim["D"], sim["g"]
    base = dict(sid=sc["sid"], block=sc["block"], n=sc["n"], G=sc["G"], rho=sc["rho"], kappa=sc["kappa"],
                prev=sc["prev"], rep=rep, ate_true=sim["ate"], prev_obs=Y.mean(), p_treat=D.mean())
    rows = []
    # pembanding 1: selisih rata-rata
    est = Y[D == 1].mean() - Y[D == 0].mean()
    psi = np.where(D == 1, (Y - Y[D == 1].mean()) / D.mean(), -(Y - Y[D == 0].mean()) / (1 - D.mean()))
    rows.append({**base, "method": "Naive", "scheme": "-", "spec": "-", "est": est,
                 "se_iid": psi.std(ddof=1) / math.sqrt(len(Y)), "se_cl": cluster_se(psi, g)})
    # pembanding 2: OLS/LPM dengan kovariat spesifikasi C
    Xc = features(sim, "C"); b, se, _ = ols_cluster(Y, np.column_stack([np.ones(len(Y)), D, Xc]), g)
    rows.append({**base, "method": "OLS", "scheme": "-", "spec": "C", "est": b[1], "se_iid": np.nan, "se_cl": se[1]})
    specs = ["A", "B", "C"] if sc["block"] == "baseline" else ["C"]   # hemat waktu: 3x3 penuh hanya di baseline
    for spec in specs:
        Xs = features(sim, spec)
        for scheme in ["random", "group", "spatial"]:
            folds = make_folds(scheme, g, sim["region_xy"], N_FOLDS, seed=rep)
            r = dml_aipw(Y, D, Xs, folds, True, g, fast=True)
            tau_hat = dr_learner(r["psi"], Xs, folds, fast=True)
            blp = blp_test(r["psi"], tau_hat, g)
            rows.append({**base, "method": "DML", "scheme": scheme, "spec": spec, "est": r["ate"],
                         "se_iid": r["se_iid"], "se_cl": r["se_cl"], "ps_extreme": r["extreme"],
                         "pehe": float(np.sqrt(np.mean((tau_hat - sim["tau"]) ** 2))),
                         "het_reject": float(blp["p_het"] < 0.05 and blp["beta_het"] > 0)})
    return rows

# ---- desain skenario ----
BASE = dict(n=2000, G=50, prev=0.05, kappa=1.0)
design = []
for rho in [0, 1, 2]:
    design.append(dict(block="baseline", **BASE, rho=rho))
    for pv in [0.03, 0.10]: design.append(dict(block="prevalence", **{**BASE, "prev": pv}, rho=rho))
    for (n_, G_) in [(1000, 35), (5000, 100)]: design.append(dict(block="size", **{**BASE, "n": n_, "G": G_}, rho=rho))
    design.append(dict(block="overlap", **{**BASE, "kappa": 2.5}, rho=rho))
for i, d in enumerate(design): d["sid"] = i
if MODE == "FAST":   # uji cepat: skenario kecil
    for d in design: d["n"] = min(d["n"], 1000); d["G"] = min(d["G"], 35)
    design = [d for d in design if d["block"] in ("baseline", "prevalence")]
design = [d for d in design if d["sid"] % SHARD[1] == SHARD[0]]
print(len(design), "skenario di shard ini")
pd.DataFrame(design)

In [ ]:
RAW = os.path.join(OUT, "sim", f"sim_raw_{MODE}_shard{SHARD[0]}of{SHARD[1]}.csv")
# Lanjutkan shard yang sama dari commit sebelumnya bila output-nya ditambahkan sebagai Input
if RUN_SIMULATION and not os.path.exists(RAW):
    prev = sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(RAW)}", recursive=True))
    if prev:
        import shutil; shutil.copy(prev[0], RAW); print("Melanjutkan dari", prev[0])
if RUN_SIMULATION:
    done = set(pd.read_csv(RAW)["sid"].unique()) if os.path.exists(RAW) else set()
    t_all = time.time()
    for sc in design:
        if sc["sid"] in done:
            print("lewati sid", sc["sid"], "(sudah ada)"); continue
        t0 = time.time()
        res = Parallel(n_jobs=N_JOBS)(delayed(one_rep)(sc, r) for r in range(N_REPS))
        df = pd.DataFrame([row for rr in res for row in rr])
        df.to_csv(RAW, mode="a", header=not os.path.exists(RAW), index=False)
        dt = time.time() - t0
        done.add(sc["sid"])
        print(f"sid={sc['sid']:2d} {sc['block']:10s} n={sc['n']} G={sc['G']} prev={sc['prev']} "
              f"kappa={sc['kappa']} rho={sc['rho']} | {dt/60:.1f} menit ({dt/N_REPS:.2f} dtk/rep)")
    print(f"Total {(time.time()-t_all)/60:.1f} menit")

### A.2 Ringkasan metrik
Bias, RMSE, **cakupan CI 95%** (SE iid dan SE klaster) + **MCSE** cakupan $=\sqrt{\hat c(1-\hat c)/R}$, rasio SE/SD empiris, proporsi propensity ekstrem, PEHE, dan daya uji heterogenitas (BLP). Bila memakai beberapa shard, gabungkan semua `sim_raw_*.csv` (sel ini otomatis membaca semuanya).

In [ ]:
import glob
# Gabungkan SEMUA shard: output sesi ini + output commit lain yang ditambahkan sebagai Input (cari rekursif,
# karena Kaggle memasang output notebook di /kaggle/input/<nama>/sim/... atau /kaggle/input/notebooks/<user>/<nama>/sim/...)
raw_files = sorted(set(glob.glob(os.path.join(OUT, "sim", f"sim_raw_{MODE}_shard*.csv")) +
                       glob.glob(f"/kaggle/input/**/sim_raw_{MODE}_shard*.csv", recursive=True)))
for f in raw_files: print("  shard:", f)
if raw_files:
    sim_raw = pd.concat([pd.read_csv(f) for f in raw_files], ignore_index=True)
    sim_raw = sim_raw.drop_duplicates(subset=["sid", "rep", "method", "scheme", "spec"])
    n_design = 18 if MODE == "FULL" else 6
    got = sorted(sim_raw["sid"].unique())
    print(f"Skenario terkumpul: {len(got)}/{n_design} -> sid {got}")
    if len(got) < n_design:
        print("!! BELUM LENGKAP. Skenario yang belum ada:", sorted(set(range(n_design)) - set(got)) if MODE == "FULL" else "")
    reps = sim_raw.groupby("sid")["rep"].nunique()
    if (reps < N_REPS).any(): print("!! Replikasi kurang pada sid:", reps[reps < N_REPS].to_dict())
else:
    sim_raw = pd.DataFrame(); print("!! Tidak ada sim_raw_*.csv yang ditemukan.")

def summarize(d):
    err = d["est"] - d["ate_true"]; R = len(d)
    out = dict(R=R, ate_true=d["ate_true"].mean(), bias=err.mean(), abs_rel_bias=abs(err.mean()) / abs(d["ate_true"].mean()),
               rmse=np.sqrt(np.mean(err ** 2)), sd_emp=d["est"].std(ddof=1), se_cl_mean=d["se_cl"].mean())
    for s in ["se_iid", "se_cl"]:
        if d[s].notna().any():
            c = np.mean(np.abs(err) <= 1.96 * d[s]); out[f"cov_{s[3:]}"] = c; out[f"mcse_cov_{s[3:]}"] = math.sqrt(c * (1 - c) / R)
    out["se_ratio"] = out["se_cl_mean"] / out["sd_emp"] if out["sd_emp"] > 0 else np.nan
    for c in ["ps_extreme", "pehe", "het_reject", "prev_obs"]:
        if c in d and d[c].notna().any(): out[c] = d[c].mean()
    return pd.Series(out)

if len(sim_raw):
    keys = ["block", "sid", "n", "G", "prev", "kappa", "rho", "method", "scheme", "spec"]
    sim_sum = pd.DataFrame([{**dict(zip(keys, k)), **summarize(grp_).to_dict()}
                            for k, grp_ in sim_raw.groupby(keys, dropna=False)])
    sim_sum.to_csv(os.path.join(OUT, "sim", f"sim_summary_{MODE}.csv"), index=False)
    show = sim_sum[(sim_sum.block == "baseline")][["rho", "method", "scheme", "spec", "bias", "rmse", "cov_cl", "mcse_cov_cl", "se_ratio", "pehe", "het_reject"]]
    try: display(show.round(4))
    except NameError: print(show.round(4).to_string())

In [ ]:
# Gambar A1: bias & cakupan vs rho (baseline), panel per spesifikasi, garis per skema cross-fitting
def fig_rho(sim_sum, block="baseline", fname="figA1_bias_coverage_rho.png"):
    d = sim_sum[(sim_sum.block == block)]
    fig, ax = plt.subplots(2, 3, figsize=(11, 6), sharex=True)
    colors = {"random": "#4C72B0", "group": "#DD8452", "spatial": "#55A868"}
    for j, spec in enumerate(["A", "B", "C"]):
        for sch, col in colors.items():
            s = d[(d.method == "DML") & (d.spec == spec) & (d.scheme == sch)].groupby("rho").mean(numeric_only=True)
            ax[0, j].plot(s.index, s["bias"], "o-", c=col, label=sch)
            ax[1, j].errorbar(s.index, s["cov_cl"], yerr=[np.minimum(1.96 * s["mcse_cov_cl"], s["cov_cl"]), np.minimum(1.96 * s["mcse_cov_cl"], 1 - s["cov_cl"])], fmt="o-", c=col, capsize=3, label=sch)
        nv = d[d.method == "Naive"].groupby("rho").mean(numeric_only=True)
        ax[0, j].plot(nv.index, nv["bias"], "k--", lw=1, label="naive")
        ax[0, j].axhline(0, c="grey", lw=.6); ax[1, j].axhline(.95, c="grey", lw=.6)
        ax[0, j].set_title(f"Spec {spec}"); ax[1, j].set_xlabel("rho (spatial confounding)")
    ax[0, 0].set_ylabel("Bias (risk difference)"); ax[1, 0].set_ylabel("95% CI coverage (cluster SE)")
    ax[0, 0].legend(frameon=False, fontsize=8); fig.tight_layout()
    fig.savefig(os.path.join(OUT, "fig", fname)); plt.show()

def fig_factor(sim_sum, block, xcol, fname, spec="C"):
    d = sim_sum[(sim_sum.block.isin([block, "baseline"])) & (sim_sum.method == "DML") & (sim_sum.spec == spec)].copy()
    if xcol == "nG": d["nG"] = d["n"].astype(str) + "/" + d["G"].astype(str)
    if d[xcol].nunique() < 2: return
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
    for j, rho in enumerate(sorted(d.rho.unique())):
        for sch, col in {"random": "#4C72B0", "group": "#DD8452", "spatial": "#55A868"}.items():
            s = d[(d.rho == rho) & (d.scheme == sch)].groupby(xcol).mean(numeric_only=True)
            ax[j].errorbar(range(len(s)), s["cov_cl"], yerr=[np.minimum(1.96 * s["mcse_cov_cl"], s["cov_cl"]), np.minimum(1.96 * s["mcse_cov_cl"], 1 - s["cov_cl"])], fmt="o-", c=col, capsize=3, label=sch)
            ax[j].set_xticks(range(len(s))); ax[j].set_xticklabels(s.index)
        ax[j].axhline(.95, c="grey", lw=.6); ax[j].set_title(f"rho = {rho}"); ax[j].set_xlabel(xcol)
    ax[0].set_ylabel("Coverage (cluster SE)"); ax[0].legend(frameon=False, fontsize=8); fig.tight_layout()
    fig.savefig(os.path.join(OUT, "fig", fname)); plt.show()

if len(sim_raw):
    fig_rho(sim_sum)
    fig_factor(sim_sum, "prevalence", "prev", "figA2_coverage_prevalence.png")
    fig_factor(sim_sum, "size", "nG", "figA3_coverage_size.png")
    fig_factor(sim_sum, "overlap", "kappa", "figA4_coverage_overlap.png")

In [ ]:
# Tabel 2 naskah: baseline, rho = 2, spesifikasi C (+ naif)
if len(sim_raw):
    b = sim_sum[(sim_sum.block == "baseline") & (sim_sum.rho == sim_sum.rho.max())]
    t2 = pd.concat([b[b.method == "Naive"], b[b.method == "OLS"],
                    b[(b.method == "DML") & (b.spec == "C")].sort_values("scheme")])
    t2 = t2[["method", "scheme", "spec", "bias", "rmse", "cov_iid", "cov_cl", "mcse_cov_cl", "se_ratio", "R"]]
    t2.to_csv(os.path.join(OUT, "tab", "T2_mc_baseline_rho2.csv"), index=False)
    print(t2.round(4).to_string(index=False))

# BAGIAN B — Analisis empiris (hasil utama naskah)
Spesifikasi utama: cross-fitting **acak** (`MAIN_SCHEME`), kovariat **C** (X + koordinat + rata-rata kab/kota ala Mundlak), SE klaster kab/kota, bobot longitudinal IFLS.

In [ ]:
COLS = dict(
    id="id", region="kab_kode", province="prov_kode", lat="lat", lon="lon",
    treat="kredit", weight="weight",
    outcomes={"usaha_tutup": "bin", "ihs_laba": "cont", "tenaga_kerja": "cont"},
    main_outcome="usaha_tutup",
    covariates=["ihs_laba07", "pekerja07", "ihs_aset07", "umur_usaha07", "n_usaha07", "usaha_di_rumah07",
                "pinjam07", "ihs_pinjam07", "log_pce07", "hhsize07", "kepala_perempuan", "usia_kepala",
                "pendidikan_kepala", "perkotaan07", "n_lembaga_kredit_desa07", "jarak_lembaga_min07"],
    cate_vars=["ihs_laba07", "umur_usaha07", "kepala_perempuan", "pendidikan_kepala",
               "perkotaan07", "pinjam07", "jarak_lembaga_min07", "log_pce07"],
    borrower="tidak_ada", default="tidak_ada",
)
HET_OUTCOMES = ["usaha_tutup", "ihs_laba"]   # heterogenitas, robustness, penargetan
if RUN_EMPIRICAL:
    assert DATA_PATH and os.path.exists(DATA_PATH), "umkm_ifls_v2.csv tidak ditemukan: jalankan Bagian P atau isi DATA_PATH"
    raw = pd.read_csv(DATA_PATH)
    COLS["covariates"] = [c for c in COLS["covariates"] if c in raw] + [c for c in raw.columns if c.endswith("_miss")]
    print(raw.shape, "| kovariat:", len(COLS["covariates"]))

In [ ]:
if RUN_EMPIRICAL:
    # B1. Validasi & persiapan
    C = COLS
    need = [C["treat"], C["region"], C["province"], C["lat"], C["lon"]] + C["covariates"]
    miss = [c for c in need if c not in raw]
    assert not miss, f"Kolom tidak ada: {miss}"
    outs = {k: v for k, v in C["outcomes"].items() if k in raw}
    dat = raw.dropna(subset=need).copy()
    dat[C["treat"]] = dat[C["treat"]].astype(int)
    assert set(dat[C["treat"]].unique()) <= {0, 1}
    W = dat[C["weight"]].values.astype(float) if C["weight"] in dat else np.ones(len(dat))
    grp = dat[C["region"]].values
    reg_xy = dat.groupby(C["region"])[[C["lon"], C["lat"]]].mean().rename(columns={C["lon"]: "x", C["lat"]: "y"})
    print(f"N={len(dat):,} | kab/kota={dat[C['region']].nunique()} | provinsi={dat[C['province']].nunique()} | "
          f"penerima KUR={np.average(dat[C['treat']], weights=W):.1%}")
    for o, t in outs.items():
        print(f"  {o:14s} ({t}) missing={dat[o].isna().mean():.1%}" + (f" prevalensi={dat[o].mean():.2%}" if t == "bin" else ""))

    def feat_matrix(df, spec):
        X = df[C["covariates"]].values.astype(float)
        if spec == "A": return X
        xy = df[[C["lon"], C["lat"]]].values.astype(float)
        if spec == "B": return np.column_stack([X, xy])
        return np.column_stack([X, xy, mundlak(X, df[C["region"]].values)])

In [ ]:
if RUN_EMPIRICAL:
    # B2. Statistik deskriptif berbobot & keseimbangan (SMD), sebelum dan sesudah IPW
    def wstats(x, w): m = np.average(x, weights=w); return m, np.sqrt(np.average((x - m) ** 2, weights=w))
    D_ = dat[C["treat"]].values
    X_C = feat_matrix(dat, "C")
    folds_main = make_folds(MAIN_SCHEME, grp, reg_xy, N_FOLDS, seed=SEED)
    e_hat = np.zeros(len(dat))
    for tr, te in folds_main:
        e_hat[te] = make_clf().fit(X_C[tr], D_[tr]).predict_proba(X_C[te])[:, 1]
    e_c = np.clip(e_hat, PS_CLIP, 1 - PS_CLIP)
    ipw = np.where(D_ == 1, 1 / e_c, 1 / (1 - e_c)) * W
    rows = []
    for v in C["covariates"] + list(outs):
        x = dat[v].values.astype(float); ok = ~np.isnan(x)
        m1, s1 = wstats(x[ok & (D_ == 1)], W[ok & (D_ == 1)]); m0, s0 = wstats(x[ok & (D_ == 0)], W[ok & (D_ == 0)])
        a1, _ = wstats(x[ok & (D_ == 1)], ipw[ok & (D_ == 1)]); a0, _ = wstats(x[ok & (D_ == 0)], ipw[ok & (D_ == 0)])
        sp = np.sqrt((s1 ** 2 + s0 ** 2) / 2) or 1
        rows.append(dict(variable=v, mean_KUR=m1, mean_nonKUR=m0, SMD_raw=(m1 - m0) / sp,
                         SMD_IPW=(a1 - a0) / sp if v in C["covariates"] else np.nan))
    tab_desc = pd.DataFrame(rows); tab_desc.to_csv(os.path.join(OUT, "tab", f"T1_descriptives{TAG}.csv"), index=False)
    print(tab_desc.round(3).to_string(index=False))

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
    ax[0].hist(e_hat[D_ == 1], bins=40, alpha=.6, density=True, label="KUR"); ax[0].hist(e_hat[D_ == 0], bins=40, alpha=.6, density=True, label="non-KUR")
    ax[0].set_xlabel("Cross-fitted propensity score"); ax[0].legend(frameon=False); ax[0].set_title("Overlap")
    t = tab_desc.dropna(subset=["SMD_IPW"]); yy = np.arange(len(t))
    ax[1].scatter(t.SMD_raw, yy, label="raw"); ax[1].scatter(t.SMD_IPW, yy, marker="x", label="IPW")
    ax[1].set_yticks(yy); ax[1].set_yticklabels(t.variable, fontsize=7); ax[1].axvline(.1, ls=":", c="grey"); ax[1].axvline(-.1, ls=":", c="grey")
    ax[1].set_xlabel("Standardized mean difference"); ax[1].legend(frameon=False)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig", f"figB1_overlap_balance{TAG}.png")); plt.show()
    print(f"Propensity di luar [0.1, 0.9]: {np.mean((e_hat<.1)|(e_hat>.9)):.1%}")

In [ ]:
if RUN_EMPIRICAL:
    # B3. Pembanding konvensional: OLS/LPM + FE provinsi, SE klaster kab/kota, + robustness value Cinelli-Hazlett
    def rv_cinelli(t, dof, q=1.0, alpha=None):
        f = q * abs(t) / math.sqrt(dof)
        return 0.5 * (math.sqrt(f ** 4 + 4 * f ** 2) - f ** 2)
    prov_d = pd.get_dummies(dat[C["province"]], drop_first=True).values.astype(float)
    ols_rows = []
    for o, typ in outs.items():
        ok = dat[o].notna().values
        Xm = np.column_stack([np.ones(ok.sum()), D_[ok], dat.loc[ok, C["covariates"]].values.astype(float), prov_d[ok]])
        b, se, G = ols_cluster(dat.loc[ok, o].values.astype(float), Xm, grp[ok], W[ok])
        tstat = b[1] / se[1]; dof = ok.sum() - Xm.shape[1]
        ols_rows.append(dict(outcome=o, coef=b[1], se=se[1], p=2 * (1 - stats.t.cdf(abs(tstat), G - 1)), RV=rv_cinelli(tstat, dof), N=ok.sum()))
    tab_ols = pd.DataFrame(ols_rows); tab_ols.to_csv(os.path.join(OUT, "tab", f"T2_ols_fe{TAG}.csv"), index=False)
    print(tab_ols.round(4).to_string(index=False))

### B4. DML-AIPW: ATE & ATT untuk setiap outcome × skema cross-fitting × spesifikasi
**Spesifikasi utama:** fold acak + spesifikasi C + SE klaster kab/kota (coverage terbaik di Monte Carlo). Fold grup & spatial-block menjadi robustness. Skema/spesifikasi lain menjadi tabel *robustness* yang dimotivasi langsung oleh hasil Monte Carlo.

In [ ]:
if RUN_EMPIRICAL:
    dml_rows, store = [], {}
    for o, typ in outs.items():
        ok = dat[o].notna().values; d_ = dat[ok]
        Yv = d_[o].values.astype(float); Dv = d_[C["treat"]].values; gv = d_[C["region"]].values; wv = W[ok]
        rxy = d_.groupby(C["region"])[[C["lon"], C["lat"]]].mean().rename(columns={C["lon"]: "x", C["lat"]: "y"})
        for spec in ["A", "B", "C"]:
            Xs = feat_matrix(d_, spec)
            for scheme in ["random", "group", "spatial"]:
                folds = make_folds(scheme, gv, rxy, N_FOLDS, seed=SEED)
                r = dml_aipw(Yv, Dv, Xs, folds, typ == "bin", gv, wv)
                dml_rows.append(dict(outcome=o, spec=spec, scheme=scheme, ATE=r["ate"], SE_cl=r["se_cl"], SE_iid=r["se_iid"],
                                     CI_lo=r["ate"] - 1.96 * r["se_cl"], CI_hi=r["ate"] + 1.96 * r["se_cl"],
                                     ATT=r["att"], ATT_SE=r["att_se"], ps_extreme=r["extreme"], N=int(ok.sum())))
                store[(o, spec, scheme)] = dict(r=r, X=Xs, folds=folds, g=gv, w=wv, idx=np.where(ok)[0])
    tab_dml = pd.DataFrame(dml_rows); tab_dml.to_csv(os.path.join(OUT, "tab", f"T3_dml_ate{TAG}.csv"), index=False)
    main = tab_dml[(tab_dml.spec == "C") & (tab_dml.scheme == MAIN_SCHEME)]
    print(f"Spesifikasi utama (C, {MAIN_SCHEME}):"); print(main.round(4).to_string(index=False))

    # Gambar: koefisien per outcome & skema
    fig, axes = plt.subplots(1, len(outs), figsize=(3.2 * len(outs), 3.2))
    axes = np.atleast_1d(axes)
    for ax, o in zip(axes, outs):
        t = tab_dml[tab_dml.outcome == o].reset_index(drop=True)
        lab = t.spec + "-" + t.scheme
        ax.errorbar(t.ATE, np.arange(len(t)), xerr=1.96 * t.SE_cl, fmt="o", capsize=2)
        ax.set_yticks(np.arange(len(t))); ax.set_yticklabels(lab, fontsize=7); ax.axvline(0, c="grey", lw=.6); ax.set_title(o)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig", f"figB2_ate_robustness{TAG}.png")); plt.show()

### B5. Heterogenitas (DR-learner → BLP, GATES, CLAN) untuk usaha tutup dan laba

In [ ]:
def heterogeneity(o):
    s = store[(o, "C", MAIN_SCHEME)]; r = s["r"]; d_ = dat.iloc[s["idx"]]
    Xcate = d_[C["cate_vars"]].values.astype(float)
    tau_hat = dr_learner(r["psi"], Xcate, s["folds"])
    blp = blp_test(r["psi"], tau_hat, s["g"], s["w"])
    gt = gates(r["psi"], tau_hat, s["g"], 5, s["w"])
    q = pd.qcut(tau_hat, 5, labels=False, duplicates="drop")
    clan = pd.DataFrame({v: [d_.loc[q == q.max(), v].mean(), d_.loc[q == 0, v].mean()] for v in C["cate_vars"]},
                        index=["Q5_highest_tau", "Q1_lowest_tau"]).T   # untuk usaha_tutup: manfaat terbesar = Q1 (tau paling negatif)
    clan["diff"] = clan.iloc[:, 0] - clan.iloc[:, 1]
    pd.DataFrame([blp]).to_csv(os.path.join(OUT, "tab", f"T6a_blp_{o}.csv"), index=False)
    gt.to_csv(os.path.join(OUT, "tab", f"T6b_gates_{o}.csv"), index=False)
    clan.to_csv(os.path.join(OUT, "tab", f"T6c_clan_{o}.csv"))
    print(f"\n== {o} | BLP slope={blp['beta_het']:.4f} (SE {blp['se_het']:.4f}, p={blp['p_het']:.3f})")
    print(gt.round(4).to_string(index=False)); print(clan.round(3))
    return dict(tau_hat=tau_hat, blp=blp, gates=gt, clan=clan)

if RUN_EMPIRICAL:
    HET = {o: heterogeneity(o) for o in HET_OUTCOMES if o in outs}
    fig, ax = plt.subplots(1, len(HET), figsize=(5 * len(HET), 3.3))
    for a, (o, h) in zip(np.atleast_1d(ax), HET.items()):
        g = h["gates"]; a.errorbar(g.group, g.gate, yerr=1.96 * g.se, fmt="o-", capsize=3)
        a.axhline(store[(o, "C", MAIN_SCHEME)]["r"]["ate"], ls="--", c="grey", label="ATE")
        a.set_xlabel("Quintile of predicted effect"); a.set_ylabel(f"GATE: {o}"); a.legend(frameon=False)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig", "figB3_gates.png")); plt.show()

    CF_RES = {}
    if HAS_ECONML:
        from econml.dml import CausalForestDML
        for o in HET:
            try:
                s = store[(o, "C", MAIN_SCHEME)]; d_ = dat.iloc[s["idx"]]
                cf = CausalForestDML(model_y=make_reg(), model_t=make_clf(), discrete_treatment=True, cv=s["folds"],
                                     n_estimators=1000, min_samples_leaf=20, random_state=SEED)
                cf.fit(d_[o].values.astype(float), d_[C["treat"]].values, X=d_[C["cate_vars"]].values.astype(float),
                       W=s["X"], sample_weight=s["w"])
                inf = cf.ate_inference(X=d_[C["cate_vars"]].values.astype(float))
                CF_RES[o] = dict(ATE_CF=float(inf.mean_point), SE_CF=float(inf.stderr_mean))
                print("Causal Forest", o, CF_RES[o])
            except Exception as ex:
                print("Causal Forest gagal untuk", o, ":", str(ex)[:150])

### B6. Robustness (Tabel 5): trimming, random forest, tanpa bobot, non-pindah, leave-one-province-out (+ Causal Forest)

In [ ]:
def rxy_of(df):
    return df.groupby(C["region"])[[C["lon"], C["lat"]]].mean().rename(columns={C["lon"]: "x", C["lat"]: "y"})

def run_dml(df, y, binary, w=None, kind="gbm", fast=False):
    f = make_folds(MAIN_SCHEME, df[C["region"]].values, rxy_of(df), N_FOLDS, seed=SEED)
    return dml_aipw(y, df[C["treat"]].values, feat_matrix(df, "C"), f, binary, df[C["region"]].values, w, kind=kind, fast=fast)

def robustness(o):
    typ = outs[o]; s = store[(o, "C", MAIN_SCHEME)]; d_ = dat.iloc[s["idx"]].reset_index(drop=True)
    Y = d_[o].values.astype(float); b = typ == "bin"; w = s["w"]
    rows = [dict(check=f"Main ({MAIN_SCHEME} folds, set C, GBM)", ATE=s["r"]["ate"], SE=s["r"]["se_cl"], N=len(Y))]
    for alt in [x for x in ("group", "spatial") if x != MAIN_SCHEME]:
        ra = store[(o, "C", alt)]["r"]; rows.append(dict(check=f"{alt.capitalize()} folds (set C)", ATE=ra["ate"], SE=ra["se_cl"], N=len(Y)))
    k = (s["r"]["e"] > .1) & (s["r"]["e"] < .9); dt = d_[k].reset_index(drop=True)
    r1 = run_dml(dt, Y[k], b, w[k]); rows.append(dict(check="Trimmed 0.1<e<0.9", ATE=r1["ate"], SE=r1["se_cl"], N=int(k.sum())))
    r2 = dml_aipw(Y, d_[C["treat"]].values, s["X"], s["folds"], b, s["g"], w, kind="rf")
    rows.append(dict(check="Random-forest learners", ATE=r2["ate"], SE=r2["se_cl"], N=len(Y)))
    r3 = dml_aipw(Y, d_[C["treat"]].values, s["X"], s["folds"], b, s["g"], None)
    rows.append(dict(check="Unweighted", ATE=r3["ate"], SE=r3["se_cl"], N=len(Y)))
    if "pindah" in d_:
        m = (d_.pindah == 0).values; dm = d_[m].reset_index(drop=True)
        r4 = run_dml(dm, Y[m], b, w[m]); rows.append(dict(check="Non-movers only", ATE=r4["ate"], SE=r4["se_cl"], N=int(m.sum())))
    if o in CF_RES:
        rows.append(dict(check="Causal Forest (econml)", ATE=CF_RES[o]["ATE_CF"], SE=CF_RES[o]["SE_CF"], N=len(Y)))
    lo = []
    for pv in sorted(d_[C["province"]].unique()):
        m = (d_[C["province"]] != pv).values; dd = d_[m].reset_index(drop=True)
        if dd[C["region"]].nunique() < 2 * N_FOLDS: continue
        lo.append(dict(dropped=pv, ATE=run_dml(dd, Y[m], b, w[m])["ate"]))
    lo = pd.DataFrame(lo)
    rows.append(dict(check="Leave one province out (min / max)", ATE=lo.ATE.min(), SE=lo.ATE.max(), N=len(lo)))
    t = pd.DataFrame(rows); t.to_csv(os.path.join(OUT, "tab", f"T5_robustness_{o}.csv"), index=False)
    lo.to_csv(os.path.join(OUT, "tab", f"T5b_lopo_{o}.csv"), index=False)
    print(f"\n== {o}"); print(t.round(4).to_string(index=False))
    return t

if RUN_EMPIRICAL:
    ROB = {o: robustness(o) for o in HET_OUTCOMES if o in outs}

### B6b. Margin ekstensif vs intensif dan sensitivitas satuan IHS (Tabel 4)
Estimasi IHS yang memuat nol bergantung pada satuan (Chen & Roth, 2024). Estimasi khusus usaha yang bertahan bersifat deskriptif (bersyarat pada outcome pasca-treatment).

In [ ]:
if RUN_EMPIRICAL and "laba_rp" in dat:
    w_all = W
    laba = dat["laba_rp"].fillna(0).values.astype(float)
    rows = []
    for sc, nm in [(1, "Rp"), (1e3, "Rp thousand"), (1e6, "Rp million")]:
        r = run_dml(dat, np.arcsinh(laba / sc), False, w_all)
        rows.append(dict(spec=f"Pooled IHS, profit in {nm}", ATE=r["ate"], SE=r["se_cl"], N=len(dat)))
    lv = laba / 1e6; lv = np.clip(lv, None, np.quantile(lv, .99))
    r = run_dml(dat, lv, False, w_all); rows.append(dict(spec="Profit level, Rp million (winsorised p99)", ATE=r["ate"], SE=r["se_cl"], N=len(dat)))
    m = (dat.usaha_tutup == 0).values; ds = dat[m].reset_index(drop=True)
    r = run_dml(ds, np.log(np.clip(laba[m], 1, None)), False, w_all[m])
    rows.append(dict(spec="Survivors only: log profit", ATE=r["ate"], SE=r["se_cl"], N=int(m.sum())))
    r = run_dml(ds, np.arcsinh(laba[m]), False, w_all[m])
    rows.append(dict(spec="Survivors only: IHS profit (Rp)", ATE=r["ate"], SE=r["se_cl"], N=int(m.sum())))
    tab_margin = pd.DataFrame(rows); tab_margin.to_csv(os.path.join(OUT, "tab", "T4_margins_scale.csv"), index=False)
    print(tab_margin.round(4).to_string(index=False))
    print("Median laba usaha bertahan (Rp juta):", round(float(np.median(laba[m]) / 1e6), 2))

### B7. Penargetan dan implikasi kebijakan (ATT vs ATE, kurva penargetan)

In [ ]:
def targeting(o):
    s = store[(o, "C", MAIN_SCHEME)]; r = s["r"]; psi = r["psi"]; w_ = s["w"]; g_ = s["g"]
    d_ = dat.iloc[s["idx"]]; Dv = d_[C["treat"]].values; tau_hat = HET[o]["tau_hat"]
    sign = -1 if outs[o] == "bin" else 1          # untuk usaha tutup, manfaat = penurunan (nilai negatif)
    order = np.argsort(-sign * tau_hat); curve = []
    for qq in np.arange(.1, 1.01, .1):
        top = order[: int(round(qq * len(psi)))]; mm = np.average(psi[top], weights=w_[top])
        curve.append(dict(share=qq, avg_effect=mm, se=cluster_se(psi[top] - mm, g_[top], w_[top])))
    curve = pd.DataFrame(curve)
    share_now = np.average(Dv, weights=w_); top_now = order[: int(round(share_now * len(psi)))]
    pol = dict(outcome=o, ATE=r["ate"], ATT=r["att"], ATT_SE=r["att_se"], current_share=share_now,
               effect_current_allocation=r["att"], effect_cate_targeting=np.average(psi[top_now], weights=w_[top_now]))
    curve.to_csv(os.path.join(OUT, "tab", f"T7_targeting_curve_{o}.csv"), index=False)
    return pol, curve

if RUN_EMPIRICAL:
    POL = {}; fig, ax = plt.subplots(1, len(HET), figsize=(5 * len(HET), 3.3))
    for a, o in zip(np.atleast_1d(ax), HET):
        pol, curve = targeting(o); POL[o] = pol
        a.plot(curve.share, curve.avg_effect, "o-", label="Target by predicted effect")
        a.fill_between(curve.share, curve.avg_effect - 1.96 * curve.se, curve.avg_effect + 1.96 * curve.se, alpha=.2)
        a.axhline(pol["ATE"], ls="--", c="grey", label="Random allocation (ATE)")
        a.scatter([pol["current_share"]], [pol["ATT"]], c="red", zorder=5, label="Current borrowers (ATT)")
        a.set_xlabel("Share of enterprises receiving credit"); a.set_ylabel(f"Average effect on {o}"); a.legend(frameon=False, fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig", "figB4_targeting.png")); plt.show()
    tab_pol = pd.DataFrame(POL.values()); tab_pol.to_csv(os.path.join(OUT, "tab", "T7b_policy.csv"), index=False)
    print(tab_pol.round(4).to_string(index=False))

# BAGIAN D — Uji plasebo 2000–2007 (Tabel 6)
Kredit 2014 tidak mungkin menyebabkan usaha tutup antara 2000 dan 2007, jadi estimasinya seharusnya nol bila kovariat pra-periode menyerap seleksi ke kredit.
Input: `umkm_ifls3_placebo.csv` (keluaran notebook builder IFLS3; tambahkan lewat *Add Input*). Kovariat = karakteristik 2000 saja.
Spesifikasi: PLR & AIPW × (fold acak, fold blok-provinsi) × (tanpa/dengan dummy provinsi); median dari 5 pengulangan cross-fitting; SE klaster kab/kota.
Dijalankan **tanpa bobot** (seperti naskah) **dan berbobot** (konsisten dengan analisis utama).
Perbaikan: `pendidikan_kepala00` di file plasebo berisi kode jenjang IFLS (bukan tahun), dikonversi dengan peta yang sama dengan Bagian P.


In [ ]:
RUN_PLACEBO = True
PLACEBO_REPS = 5 if MODE == "FULL" else 1
TAB_PLACEBO = None

def _folds_by_label(labels, k, seed):
    u = np.unique(labels); rng = np.random.default_rng(seed); perm = rng.permutation(len(u))
    f_of = {lab: perm[i] % k for i, lab in enumerate(u)}
    f = np.array([f_of[l] for l in labels])
    return [(np.where(f != j)[0], np.where(f == j)[0]) for j in range(k) if (f == j).any()]

def _plr(Y, D, X, folds, groups, w):
    """Partially linear DML (Robinson): Y - l(X) = theta (D - m(X)) + e."""
    n = len(Y); mhat = np.zeros(n); lhat = np.zeros(n)
    for tr, te in folds:
        mhat[te] = _fit_pred(make_clf(), X[tr], D[tr], X[te], True)
        lhat[te] = _fit_pred(make_clf(), X[tr], Y[tr], X[te], True)     # Y biner -> E[Y|X] via classifier
    v = D - mhat; u = Y - lhat; den = np.sum(w * v * v)
    th = np.sum(w * v * u) / den
    phi = w * v * (u - th * v) / den
    s = pd.Series(phi).groupby(np.asarray(groups)).sum().values; G = len(s)
    return th, math.sqrt(G / (G - 1) * np.sum(s ** 2))

def _median_agg(est, se):
    est, se = np.asarray(est), np.asarray(se); m = np.median(est)
    return m, np.median(np.sqrt(se ** 2 + (est - m) ** 2))     # Chernozhukov et al. (2018)

if RUN_EMPIRICAL and RUN_PLACEBO:
    hits = sorted(glob.glob("/kaggle/input/**/umkm_ifls3_placebo.csv", recursive=True) +
                  glob.glob(os.path.join(OUT, "**", "umkm_ifls3_placebo.csv"), recursive=True) +
                  glob.glob("./**/umkm_ifls3_placebo.csv", recursive=True))
    if not hits:
        print("!! umkm_ifls3_placebo.csv tidak ditemukan di Input -> Bagian D dilewati.")
    else:
        pl = pd.read_csv(hits[0], dtype={"id": str, "hhid00": str, "hhid07": str, "hhid14": str})
        EDU_CODE = {1: 0, 90: 0, 2: 6, 11: 6, 17: 6, 72: 6, 14: 6, 3: 9, 4: 9, 12: 9, 73: 9,
                    5: 12, 6: 12, 15: 12, 74: 12, 60: 14, 13: 16, 61: 16, 62: 18, 63: 21, 10: 0}
        if pl["pendidikan_kepala00"].max() > 25:
            pl["pendidikan_kepala00"] = pl["pendidikan_kepala00"].map(EDU_CODE)
            pl["pendidikan_kepala00"] = pl["pendidikan_kepala00"].fillna(pl["pendidikan_kepala00"].median())
            print("pendidikan_kepala00: kode jenjang IFLS -> tahun sekolah")
        pcov = [c for c in pl.columns if (c.endswith("00") or c.endswith("00_miss"))
                and c not in ("prov00", "kab00") and not c.startswith("hhid")]
        Xp = pl[pcov].values.astype(float)
        Xp_pd = np.column_stack([Xp, pd.get_dummies(pl["prov00"], drop_first=True).values.astype(float)])
        Yp = pl["usaha_tutup_0007"].values.astype(float); Dp = pl["kredit"].values.astype(int)
        gp = (pl["prov00"] * 100 + pl["kab00"]).values; pp = pl["prov00"].values
        print(f"Plasebo: N={len(pl):,} | kab/kota={len(np.unique(gp))} | provinsi={len(np.unique(pp))} | "
              f"kredit={Dp.mean():.1%} | tutup 2000-07={Yp.mean():.1%} | kovariat={len(pcov)}")
        rows = []
        for wlab, wv in (("unweighted", np.ones(len(pl))), ("weighted", pl["weight"].values.astype(float))):
            # selisih mentah
            m1 = np.average(Yp[Dp == 1], weights=wv[Dp == 1]); m0 = np.average(Yp[Dp == 0], weights=wv[Dp == 0])
            phi = np.where(Dp == 1, wv * (Yp - m1) / wv[Dp == 1].sum(), -wv * (Yp - m0) / wv[Dp == 0].sum())
            s = pd.Series(phi).groupby(gp).sum().values; se = math.sqrt(len(s) / (len(s) - 1) * np.sum(s ** 2))
            rows.append(dict(weights=wlab, specification="Raw difference (no controls)", estimator="Difference",
                             estimate=m1 - m0, se=se))
            for XX, xlab in ((Xp, "2000 covariates"), (Xp_pd, "+ province dummies")):
                for scheme, slab in (("random", "random folds"), ("province", "province-block folds")):
                    for est in ("PLR", "AIPW"):
                        ee, ss = [], []
                        for r in range(PLACEBO_REPS):
                            sd = SEED + 101 * r
                            folds = (list(KFold(N_FOLDS, shuffle=True, random_state=sd).split(Xp)) if scheme == "random"
                                     else _folds_by_label(pp, N_FOLDS, sd))
                            if est == "PLR":
                                a, b = _plr(Yp, Dp, XX, folds, gp, wv)
                            else:
                                o = dml_aipw(Yp, Dp, XX, folds, True, gp, wv); a, b = o["ate"], o["se_cl"]
                            ee.append(a); ss.append(b)
                        a, b = _median_agg(ee, ss)
                        rows.append(dict(weights=wlab, specification=f"{xlab}, {slab}", estimator=est, estimate=a, se=b))
        TAB_PLACEBO = pd.DataFrame(rows)
        TAB_PLACEBO["p_value"] = 2 * (1 - stats.norm.cdf(abs(TAB_PLACEBO.estimate / TAB_PLACEBO.se)))
        TAB_PLACEBO.to_csv(os.path.join(OUT, "tab", "T6_placebo.csv"), index=False)
        print(TAB_PLACEBO.round(4).to_string(index=False))


# BAGIAN E — Spesifikasi *history-augmented* (Sect. 6.5)
Untuk RT yang punya usaha di 2000 **dan** 2007 (tercocokkan lewat `hhid14`), spesifikasi utama (spatial-block, set C, berbobot) diestimasi ulang pada sampel yang sama, **tanpa** dan **dengan** riwayat usaha & pinjaman tahun 2000 sebagai kovariat tambahan. Bila estimasi hampir tidak berubah, riwayat pra-2007 tidak menyerap seleksi tambahan.


In [ ]:
TAB_HIST = None
if RUN_EMPIRICAL and RUN_PLACEBO and TAB_PLACEBO is not None:
    def _key(s): return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")
    hist = [c for c in ["ihs_omzet00", "ihs_laba00", "pekerja00", "ihs_aset00", "umur_usaha00", "n_usaha00",
                        "pinjam_nonkel00", "pinjam00", "ihs_pinjam00"] if c in pl]
    hist += [c + "_miss" for c in hist if c + "_miss" in pl]
    ph = pl[pl["usaha_tutup_0007"] == 0].assign(_k=_key(pl["hhid14"]))[["_k"] + hist].drop_duplicates("_k")
    dh = dat.assign(_k=_key(dat["hhid14"])).merge(ph, on="_k", how="inner").reset_index(drop=True)
    print(f"RT dengan usaha 2000 & 2007 yang cocok ke sampel utama: {len(dh):,} dari {len(dat):,}")
    if len(dh) < 300 or dh[C["region"]].nunique() < 2 * N_FOLDS:
        print("!! Kecocokan terlalu sedikit (cek format hhid14) -> Bagian E dilewati.")
    else:
        rows = []
        for o in [x for x in HET_OUTCOMES if x in outs]:
            do = dh[dh[o].notna()].reset_index(drop=True)
            fo = make_folds(MAIN_SCHEME, do[C["region"]].values, rxy_of(do), N_FOLDS, seed=SEED)
            XC = feat_matrix(do, "C"); XH = np.column_stack([XC, do[hist].values.astype(float)])
            Y = do[o].values.astype(float); b = outs[o] == "bin"
            for XX, lab in ((XC, "Set C, same sample"), (XH, "Set C + 2000 history")):
                r = dml_aipw(Y, do[C["treat"]].values, XX, fo, b, do[C["region"]].values,
                             do[C["weight"]].values.astype(float))
                rows.append(dict(outcome=o, specification=lab, ATE=r["ate"], SE=r["se_cl"], ATT=r["att"],
                                 ATT_SE=r["att_se"], N=len(do)))
        TAB_HIST = pd.DataFrame(rows)
        TAB_HIST.to_csv(os.path.join(OUT, "tab", "T7_history_augmented.csv"), index=False)
        print(TAB_HIST.round(4).to_string(index=False))


# BAGIAN C — Ekspor tabel (LaTeX) dan ringkasan angka untuk naskah
`angka_untuk_naskah.md` memuat semua angka yang diganti di dokumen draf (Tabel 1–5, BLP, ATT/ATE, Monte Carlo).

In [ ]:
def to_tex(df_, name, cap):
    try:
        tex = df_.to_latex(index=False, float_format="%.4f", caption=cap, label=f"tab:{name}")
    except Exception:
        tex = df_.round(4).to_string(index=False)
    open(os.path.join(OUT, "tab", f"{name}.tex"), "w").write(tex)

lines = [f"# Angka untuk naskah (MODE={MODE}, N_REPS={N_REPS})", ""]
def block(title, df_):
    lines.extend([f"## {title}", "", "```", df_.round(4).to_string(index=False), "```", ""])

if len(sim_raw):
    to_tex(sim_sum[sim_sum.method.isin(["DML", "Naive", "OLS"])][["block", "rho", "prev", "n", "G", "kappa", "method", "scheme", "spec", "bias", "rmse", "cov_iid", "cov_cl", "mcse_cov_cl", "se_ratio"]],
           "TA1_simulation", "Monte Carlo performance, all scenarios")
    block("Tabel 2 - Monte Carlo baseline, rho=2", t2)
    for blk in ["prevalence", "size", "overlap"]:
        d0 = sim_sum[(sim_sum.block.isin([blk, "baseline"])) & (sim_sum.method == "DML") & (sim_sum.spec == "C")]
        block(f"Monte Carlo - blok {blk} (spec C)", d0[["rho", "prev", "n", "G", "kappa", "scheme", "bias", "rmse", "cov_cl", "mcse_cov_cl", "se_ratio"]])
if RUN_EMPIRICAL:
    to_tex(tab_desc, "T1_descriptives", "Pre-treatment covariates and outcomes by treatment status")
    t3 = tab_dml[tab_dml.spec == "C"].pivot_table(index="outcome", columns="scheme", values=["ATE", "SE_cl"]).round(4)
    t3.columns = [f"{a}_{b}" for a, b in t3.columns]; t3 = t3.reset_index()
    att = tab_dml[(tab_dml.spec == "C") & (tab_dml.scheme == MAIN_SCHEME)][["outcome", "ATT", "ATT_SE"]]
    t3 = t3.merge(att, on="outcome").merge(tab_ols[["outcome", "coef", "se", "RV"]].rename(columns={"coef": "OLS", "se": "OLS_SE"}), on="outcome")
    t3.to_csv(os.path.join(OUT, "tab", "T3_main.csv"), index=False); to_tex(t3, "T3_main", "Effect of non-family credit on 2014 outcomes")
    to_tex(tab_dml, "TA2_dml_all", "All DML estimates by outcome, fold design and covariate set")
    block("Tabel 1 - deskriptif & keseimbangan", tab_desc)
    block("Tabel 3 - efek utama (spec C) + ATT + OLS", t3)
    block("Lampiran A2 - semua estimasi DML", tab_dml[["outcome", "spec", "scheme", "ATE", "SE_cl", "SE_iid", "ATT", "ATT_SE", "ps_extreme"]])
    if "tab_margin" in dir():
        to_tex(tab_margin, "T4_margins_scale", "Profit margins and unit sensitivity"); block("Tabel 4 - margin & sensitivitas satuan", tab_margin)
    for o, t in ROB.items():
        to_tex(t, f"T5_robustness_{o}", f"Robustness: {o}"); block(f"Tabel 5 - robustness {o}", t)
    for o, h in HET.items():
        block(f"Heterogenitas {o} - BLP", pd.DataFrame([h["blp"]])); block(f"GATES {o}", h["gates"])
        block(f"CLAN {o}", h["clan"].reset_index().rename(columns={"index": "variable"}))
    block("Kebijakan - ATT vs ATE dan penargetan", tab_pol)
    block("Ukuran sampel", pd.DataFrame([dict(N=len(dat), districts=dat[C["region"]].nunique(), provinces=dat[C["province"]].nunique(),
                                              treated=float(np.average(dat[C["treat"]], weights=W)), closure=float(dat.usaha_tutup.mean()),
                                              closure_untreated=float(dat.loc[dat[C["treat"]] == 0, "usaha_tutup"].mean()))]))
if "TAB_PLACEBO" in dir() and TAB_PLACEBO is not None:
    to_tex(TAB_PLACEBO, "T6_placebo", "Placebo: effect of 2014 credit on enterprise closure 2000-2007")
    block("Tabel 6 - plasebo (unweighted = versi naskah; weighted = konsisten dgn analisis utama)", TAB_PLACEBO)
if "TAB_HIST" in dir() and TAB_HIST is not None:
    to_tex(TAB_HIST, "T7_history_augmented", "History-augmented specification")
    block("Sect. 6.5 - history-augmented", TAB_HIST)
open(os.path.join(OUT, "draft", "angka_untuk_naskah.md"), "w").write("\n".join(lines))
print("\n".join(lines)[:6000])

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/tmp/results_{MODE}_shard{SHARD[0]}of{SHARD[1]}", "zip", OUT, ".")
zip_path = shutil.move(zip_path, os.path.join(OUT, os.path.basename(zip_path)))
print("Selesai ->", zip_path)